# 🚗 Vehicle Fuel Efficiency Prediction
## Notebook 5: Feature Engineering

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
df = pd.read_csv('../data/auto-mpg-cleaned.csv')
print(f'Loaded: {df.shape}')
df.head(3)

## 5.1 Domain-Driven Features

In [ ]:
df['power_to_weight'] = df['horsepower'] / df['weight']
df['displacement_per_cyl'] = df['displacement'] / df['cylinders']
df['weight_class'] = pd.cut(
    df['weight'],
    bins=[0, 2500, 3500, 4500, 10000],
    labels=['Light', 'Mid', 'Heavy', 'Very Heavy']
)
df['era'] = pd.cut(
    df['model_year'],
    bins=[69, 73, 78, 83],
    labels=['Pre-Crisis (70-73)', 'Crisis (74-78)', 'Post-Crisis (79-82)']
)
df['log_weight'] = np.log(df['weight'])
df['log_displacement'] = np.log(df['displacement'])
df['log_horsepower'] = np.log(df['horsepower'])
df['year_actual'] = df['model_year'] + 1900
print('New features added:')
new_features = ['power_to_weight', 'displacement_per_cyl', 'weight_class',
                'era', 'log_weight', 'log_displacement', 'log_horsepower', 'year_actual']
display(df[new_features].describe())

## 5.2 Encode Categorical Features

In [ ]:
df['origin'] = df['origin'].astype(int)
origin_dummies = pd.get_dummies(df['origin'], prefix='origin', drop_first=True, dtype=int)
df = pd.concat([df, origin_dummies], axis=1)
weight_order = {'Light': 0, 'Mid': 1, 'Heavy': 2, 'Very Heavy': 3}
df['weight_class_enc'] = df['weight_class'].map(weight_order)
era_order = {'Pre-Crisis (70-73)': 0, 'Crisis (74-78)': 1, 'Post-Crisis (79-82)': 2}
df['era_enc'] = df['era'].map(era_order)
top_brands = df['brand'].value_counts().head(10).index
df['brand_enc'] = df['brand'].where(df['brand'].isin(top_brands), other='other')
brand_dummies = pd.get_dummies(df['brand_enc'], prefix='brand', drop_first=True, dtype=int)
df = pd.concat([df, brand_dummies], axis=1)
print(f'Shape after encoding: {df.shape}')

## 5.3 New Feature Correlations with MPG

In [ ]:
new_num_features = ['power_to_weight', 'displacement_per_cyl',
                    'log_weight', 'log_displacement', 'log_horsepower']
corr_new = df[['mpg'] + new_num_features].corr()['mpg'].drop('mpg').sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_new]
ax.barh(corr_new.index, corr_new.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation of Engineered Features with MPG', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../plots/09_engineered_feature_corr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Select Final Feature Set

In [ ]:
FEATURE_COLS = [
    'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year',
    'power_to_weight', 'displacement_per_cyl', 'log_weight', 'log_displacement', 'log_horsepower',
    'weight_class_enc', 'era_enc',
    'origin_2', 'origin_3',
]
brand_cols = [c for c in df.columns if c.startswith('brand_')]
FEATURE_COLS += brand_cols
TARGET = 'mpg'
df_model = df[FEATURE_COLS + [TARGET]].dropna()
print(f'Final dataset for modeling: {df_model.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
df_model.to_csv('../data/auto-mpg-features.csv', index=False)
print('\nFeature dataset saved ✓')